# 07 · TTF Forward Curve & Storage Analysis

**Data sources:**
- EU storage: `data/processed/eu_aggregate_full.parquet` (from notebook 01)
- TTF forward curve: `data/raw/ttf_curve.csv` (from Databento NDEX.IMPACT)

**Run notebook 01 first to generate the storage parquet.**
---

In [ ]:
import sys, os, warnings
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from datetime import date, timedelta
import calendar
warnings.filterwarnings('ignore')

# ── Find project root ─────────────────────────────────────────────────────────
for _c in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (_c / 'src' / 'agsi_client').exists():
        sys.path.insert(0, str(_c))
        os.chdir(_c)
        print(f"✅ Root: {_c}")
        break

# ── Load storage data ─────────────────────────────────────────────────────────
df_eu_full = pd.read_parquet("data/processed/eu_aggregate_full.parquet")
df_eu_full.index = pd.to_datetime(df_eu_full.index).tz_localize(None)
df_eu_full = df_eu_full.sort_index()

# ── Analysis config ───────────────────────────────────────────────────────────
ANALYSIS_DATE = df_eu_full['full'].dropna().index[-1].date()  # last data point
START_DATE    = "2020-01-01"                                   # chart window

df_eu = df_eu_full[df_eu_full.index >= START_DATE].copy()
df_eu = df_eu[df_eu.index <= pd.Timestamp(ANALYSIS_DATE)]

# Core scalars — always from raw columns
fill_pct    = float(df_eu['gasInStorage'].dropna().iloc[-1] /
                    df_eu['workingGasVolume'].dropna().iloc[-1] * 100)
current_gwh = float(df_eu['gasInStorage'].dropna().iloc[-1]) * 1000     # TWh → GWh
capacity    = float(df_eu['workingGasVolume'].dropna().iloc[-1]) * 1000  # TWh → GWh
latest_date = df_eu['gasInStorage'].dropna().index[-1]

print(f"✅ Storage: {len(df_eu_full)} rows | {df_eu_full.index.min().date()} → {latest_date.date()}")
print(f"   Analysis date  : {ANALYSIS_DATE}")
print(f"   Fill rate      : {fill_pct:.1f}%")
print(f"   Storage        : {current_gwh/1000:.1f} TWh")
print(f"   Capacity       : {capacity/1000:.1f} TWh")

## 1 · TTF Forward Curve Data

In [ ]:
TTF_PATH = Path("data/raw/ttf_curve.csv")

def build_synthetic_ttf(storage_df):
    idx  = storage_df.index
    fill = storage_df['full'].reindex(idx).ffill()
    pre  = idx < '2022-01-01'
    base = pd.Series(index=idx, dtype=float)
    base[pre]  = 25 - 0.20 * fill[pre]  + np.random.normal(0, 1.5, pre.sum())
    base[~pre] = 60 - 0.55 * fill[~pre] + np.random.normal(0, 4.0, (~pre).sum())
    base = base.clip(5, 200)
    seasonal = {1:1.15,2:1.10,3:1.02,4:0.95,5:0.92,6:0.93,
                7:0.95,8:0.97,9:1.00,10:1.05,11:1.12,12:1.18}
    curve = pd.DataFrame(index=idx)
    for m in range(1, 25):
        dm     = (idx.month + m - 1) % 12 + 1
        factor = np.array([seasonal[d] for d in dm])
        curve[f'M{m}'] = (base * factor).round(2)
    curve.index.name = 'date'
    return curve

if TTF_PATH.exists():
    ttf = pd.read_csv(TTF_PATH, index_col=0, parse_dates=True)
    ttf.index = pd.to_datetime(ttf.index).tz_localize(None)
    ttf.columns = [c.strip() for c in ttf.columns]
    for col in ttf.columns:
        ttf[col] = pd.to_numeric(ttf[col], errors='coerce')
    ttf = ttf.sort_index()
    # Align with storage date range
    ttf = ttf[ttf.index >= START_DATE]
    ttf = ttf[ttf.index <= pd.Timestamp(ANALYSIS_DATE)]
    print(f"✅ TTF loaded from CSV: {ttf.shape}")
    print(f"   Date range: {ttf.index.min().date()} → {ttf.index.max().date()}")
    print(f"   Latest M1 : €{ttf['M1'].dropna().iloc[-1]:.2f}/MWh")
    print(f"   Latest M12: €{ttf['M12'].dropna().iloc[-1]:.2f}/MWh")
else:
    print("⚠️  No TTF CSV — using synthetic curve")
    ttf = build_synthetic_ttf(df_eu)
ttf.tail(3)

## 2 · Forward Curve Structure

In [ ]:
def plot_forward_curve(ttf_df, plot_date=None):
    if plot_date is None:
        row       = ttf_df.dropna(how='all').iloc[-1]
        plot_date = ttf_df.dropna(how='all').index[-1]
    else:
        plot_date = pd.Timestamp(plot_date)
        row       = ttf_df.loc[ttf_df.index <= plot_date].iloc[-1]
        plot_date = ttf_df.loc[ttf_df.index <= plot_date].index[-1]

    x_labels, y_vals, colors = [], [], []
    season_colors = {12:'#d62728',1:'#d62728',2:'#d62728',3:'#d62728',
                     4:'#2ca02c',5:'#2ca02c',6:'#2ca02c',
                     7:'#ff7f0e',8:'#ff7f0e',9:'#ff7f0e',
                     10:'#9467bd',11:'#9467bd'}
    for m in range(1, 25):
        col = f'M{m}'
        if col not in row.index or pd.isna(row[col]):
            continue
        dm   = (plot_date.month + m - 1) % 12 + 1
        yr   = plot_date.year + (plot_date.month + m - 1) // 12
        x_labels.append(f"{calendar.month_abbr[dm]}'{str(yr)[-2:]}")
        y_vals.append(float(row[col]))
        colors.append(season_colors[dm])

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=x_labels, y=y_vals, mode='lines+markers',
        line=dict(color='#2E75B6', width=2.5),
        marker=dict(size=9, color=colors, line=dict(color='white', width=1.5)),
        hovertemplate='%{x}<br>€%{y:.2f}/MWh<extra></extra>',
    ))

    # Quarter annotations
    prev_q = None
    for i, label in enumerate(x_labels):
        dm = ((plot_date.month + i) % 12) + 1
        q  = (dm - 1) // 3 + 1
        if q != prev_q:
            fig.add_vline(x=i-0.5, line_dash='dot', line_color='lightgrey', line_width=1)
            fig.add_annotation(x=i, y=max(y_vals)*1.04,
                                text=f"Q{q}", showarrow=False,
                                font=dict(size=9, color='#666'))
            prev_q = q

    m1  = y_vals[0]
    m12 = y_vals[11] if len(y_vals) > 11 else y_vals[-1]
    sw  = np.nanmean([y_vals[i] for i in range(9,12) if i < len(y_vals)]) -           np.nanmean([y_vals[i] for i in range(3,9)  if i < len(y_vals)])

    fig.update_layout(
        title=f"TTF Forward Curve — {plot_date.strftime('%d %b %Y')}  "
              f"| {'BACKWARDATION' if m1>m12 else 'CONTANGO'}  "
              f"| M1–M12: €{m1-m12:+.2f}  | W–S: €{sw:+.2f}/MWh",
        xaxis_title="Delivery month",
        yaxis_title="€/MWh",
        template="plotly_white",
        height=430,
    )
    fig.show()
    print(f"  M1:     €{m1:.2f}/MWh")
    print(f"  M12:    €{m12:.2f}/MWh")
    print(f"  W–S spread (Q4–Q3): €{sw:.2f}/MWh")

plot_forward_curve(ttf)

In [ ]:
def aggregate_curve(ttf_df):
    """Aggregate M1-M24 to M+1/2/3 + calendar quarters + seasons."""
    rows = []
    for date, row in ttf_df.iterrows():
        r = {'date': date}
        prices_by_q = {1:[],2:[],3:[],4:[]}
        for m in range(1, 25):
            col = f'M{m}'
            if col not in row.index or pd.isna(row[col]):
                continue
            price = float(row[col])
            if m <= 3:
                r[f'M+{m}'] = price
            dm = (date.month + m - 1) % 12 + 1
            cq = (dm - 1) // 3 + 1
            prices_by_q[cq].append(price)
        for q in range(1,5):
            vals = [v for v in prices_by_q[q] if not np.isnan(v)]
            r[f'Q{q}'] = round(np.mean(vals), 3) if vals else np.nan
        summer = [v for q in [2,3] for v in prices_by_q[q] if not np.isnan(v)]
        winter = [v for q in [4,1] for v in prices_by_q[q] if not np.isnan(v)]
        r['Summer'] = round(np.mean(summer), 3) if summer else np.nan
        r['Winter'] = round(np.mean(winter), 3) if winter else np.nan
        rows.append(r)
    df = pd.DataFrame(rows).set_index('date')
    df.index = pd.to_datetime(df.index)
    return df

curve_agg = aggregate_curve(ttf)
print("Aggregated columns:", curve_agg.columns.tolist())
print(f"\nLatest ({curve_agg.index[-1].date()}):")
for col in ['M+1','M+2','M+3','Q1','Q2','Q3','Q4','Summer','Winter']:
    if col in curve_agg.columns and not pd.isna(curve_agg[col].iloc[-1]):
        print(f"  {col:<8} €{curve_agg[col].iloc[-1]:.2f}/MWh")

sw = float(curve_agg['Winter'].dropna().iloc[-1]) - float(curve_agg['Summer'].dropna().iloc[-1])
print(f"\n  Winter–Summer spread: €{sw:.2f}/MWh")
print(f"  Signal: {'INJECT ✅ (W–S > €5)' if sw > 5 else 'CAUTIOUS ⚠️ (W–S 0–5)' if sw > 0 else 'NEGATIVE ❌'}") 

In [ ]:
sw_spread = curve_agg['Winter'] - curve_agg['Summer']

def _ws_hover_label(dt):
    s_mo, w_mo = [], []
    for _m in range(1, 25):
        _dm  = (dt.month + _m - 1) % 12 + 1
        _yr  = dt.year  + (dt.month + _m - 1) // 12
        _tag = f"{calendar.month_abbr[_dm]}'{str(_yr)[-2:]}"
        if _dm in [4,5,6,7,8,9]    and len(s_mo) < 6: s_mo.append(_tag)
        if _dm in [10,11,12,1,2,3]  and len(w_mo) < 6: w_mo.append(_tag)
        if len(s_mo) >= 6 and len(w_mo) >= 6: break
    s_lbl = f"{s_mo[0]}–{s_mo[-1]}" if s_mo else "—"
    w_lbl = f"{w_mo[0]}–{w_mo[-1]}" if w_mo else "—"
    return f"Winter ({w_lbl}) – Summer ({s_lbl})"

_sw_hover = np.array([_ws_hover_label(dt) for dt in sw_spread.index])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=sw_spread.index, y=sw_spread,
    mode='lines', fill='tozeroy',
    fillcolor='rgba(46,117,182,0.12)',
    line=dict(color='#2E75B6', width=1.5),
    name='Winter–Summer Spread',
    customdata=_sw_hover,
    hovertemplate='%{x|%d %b %Y}<br>%{customdata}<br>€%{y:+.2f}/MWh<extra></extra>',
))
fig.add_hline(y=0, line_color='black', line_width=1)
fig.add_hline(y=5, line_dash='dot', line_color='green',
              annotation_text='Injection breakeven (~€5/MWh)')

one_yr = sw_spread.loc[sw_spread.index >= sw_spread.index[-1] - pd.DateOffset(years=1)].mean()
fig.update_layout(
    title="TTF Winter–Summer Spread (Q4+Q1 minus Q2+Q3, €/MWh)",
    xaxis_title="Date", yaxis_title="Spread (€/MWh)",
    template="plotly_white", hovermode="x unified",
)
fig.show()

print(f"  Current : €{float(sw_spread.iloc[-1]):.2f}/MWh")
print(f"  1yr avg : €{one_yr:.2f}/MWh")

## 3 · Storage vs. TTF Correlation

In [ ]:
from src.analysis import rolling_correlation, storage_price_regression

# Merge storage fill % with TTF M1
merged = (df_eu[['full']]
          .join(ttf[['M1']].rename(columns={'M1':'ttf_front'}), how='inner')
          .dropna())
merged['year'] = merged.index.year
print(f"Merged: {len(merged)} rows | {merged.index.min().date()} → {merged.index.max().date()}")

# Rolling correlation
corr = rolling_correlation(merged, 'full', 'ttf_front', windows=[30,60,90])

fig = go.Figure()
for col, color, width in [('corr_60d','#2E75B6',2),('corr_90d','#1f4e79',1.2)]:
    fig.add_trace(go.Scatter(x=corr.index, y=corr[col], name=col,
                             line=dict(color=color, width=width)))
fig.add_hline(y=0, line_color='black', line_width=0.8)
fig.add_hline(y=-0.7, line_dash='dot', line_color='red',
              annotation_text='Strong negative (-0.7)')
fig.update_layout(title="Rolling Correlation: EU Fill % vs TTF M1",
                  yaxis=dict(range=[-1.1,1.1], title="Pearson r"),
                  template="plotly_white", hovermode="x unified")
fig.show()

current_corr = float(corr['corr_60d'].dropna().iloc[-1])
print(f"  60-day correlation: {current_corr:.2f}")
print(f"  Regime: {'STORAGE-DRIVEN' if current_corr < -0.7 else 'MIXED' if current_corr < -0.4 else 'OTHER FACTORS'}")

In [ ]:
# OLS regression
reg = storage_price_regression(merged, log_price=True)

print("OLS: log(TTF M1) ~ α + β × fill%")
print(f"  α = {reg['intercept']:.4f}  β = {reg['slope']:.4f}  R² = {reg['r_squared']:.3f}")
print(f"  {reg['interpretation']}")
print(f"\n  Price sensitivity at current fill ({fill_pct:.1f}%):")
for fp in [30,50,70,90]:
    p = np.exp(reg['intercept'] + reg['slope'] * fp)
    s = p * reg['slope']
    print(f"  Fill={fp}% → €{p:.1f}/MWh | 1pp change ≈ €{abs(s):.2f}/MWh")

In [ ]:
# 3D scatter: TTF price ~ fill rate × delivery month
from sklearn.linear_model import LinearRegression

rows = []
for m in range(1, 13):
    col = f'M{m}'
    if col not in ttf.columns: continue
    sub = ttf[[col]].join(df_eu[['full']], how='inner').dropna()
    sub['delivery_month'] = (sub.index.month + m - 1) % 12 + 1
    sub['price']          = sub[col].astype(float)
    sub['fill']           = sub['full'].astype(float)
    sub['year']           = sub.index.year
    rows.append(sub[['fill','delivery_month','price','year']])

df3d = pd.concat(rows).dropna()

fig = go.Figure()
colorscale = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b','#17becf']
for i, year in enumerate(sorted(df3d['year'].unique())):
    sub = df3d[df3d['year']==year].sample(min(300,len(df3d[df3d['year']==year])), random_state=42)
    fig.add_trace(go.Scatter3d(
        x=sub['fill'], y=sub['delivery_month'], z=sub['price'],
        mode='markers', name=str(year),
        marker=dict(size=2.5, opacity=0.5, color=colorscale[i%len(colorscale)]),
        hovertemplate='Fill:%{x:.1f}% Month:%{y} €%{z:.2f}<extra></extra>',
    ))
fig.update_layout(
    title="TTF Price Surface: Fill Rate × Delivery Month",
    scene=dict(
        xaxis=dict(title="EU Fill %", ticksuffix="%"),
        yaxis=dict(title="Delivery Month", tickmode='array',
                   tickvals=list(range(1,13)),
                   ticktext=[calendar.month_abbr[i] for i in range(1,13)]),
        zaxis=dict(title="€/MWh"),
        camera=dict(eye=dict(x=1.8, y=-1.8, z=0.8)),
    ),
    height=600, template="plotly_white",
)
fig.show()

## 4 · Injection Pace & Projections

In [ ]:
from src.analysis import required_daily_injection

def next_date(month, day, from_date):
    d = from_date.replace(month=month, day=day)
    if d <= from_date: d = d.replace(year=d.year+1)
    return d

oct1 = next_date(10, 1, ANALYSIS_DATE)
nov1 = next_date(11, 1, ANALYSIS_DATE)
apr1 = next_date(4,  1, ANALYSIS_DATE)
if apr1 <= nov1: apr1 = apr1.replace(year=apr1.year+1)

# Historical injection rates — active injection days only (Apr–Sep, inj > 0)
inj_hist   = df_eu_full['injection'].dropna()
wit_hist   = df_eu_full['withdrawal'].dropna()
inj_active = inj_hist[inj_hist.index.month.isin(range(4,10)) & (inj_hist > 0)]
wit_active = wit_hist[wit_hist.index.month.isin([10,11,12,1,2,3]) & (wit_hist > 0)]

rates = {
    'Low':  {'inj': float(inj_active.quantile(0.10)),
             'wit': float(wit_active.quantile(0.90))},
    'Avg':  {'inj': float(inj_active.quantile(0.50)),
             'wit': float(wit_active.quantile(0.50))},
    'High': {'inj': float(inj_active.quantile(0.90)),
             'wit': float(wit_active.quantile(0.10))},
}

def project_storage(start_gwh, cap_gwh, rates_dict, start_date, end_date):
    dates   = pd.date_range(start_date, end_date, freq='D')
    storage = [start_gwh]
    for d in dates[1:]:
        prev  = storage[-1]
        delta = +rates_dict['inj'] if d.month in range(4,10) else -rates_dict['wit']
        storage.append(max(0.0, min(cap_gwh, prev + delta)))
    return pd.Series(storage, index=dates, name='storage_gwh')

projections = {name: project_storage(current_gwh, capacity, r, ANALYSIS_DATE, apr1)
               for name, r in rates.items()}

def get_fill(proj, d):
    ts = pd.Timestamp(d)
    return float(proj.loc[proj.index <= ts].iloc[-1]) / capacity * 100            if (proj.index <= ts).any() else np.nan

print(f"  Start : {ANALYSIS_DATE}  fill={fill_pct:.1f}%  {current_gwh/1000:.1f} TWh")
print(f"  Oct 1 : {oct1}  Nov 1 : {nov1}  Apr 1 : {apr1}")
print(f"\n  {'Scenario':<8} {'Oct 1':>10} {'Nov 1':>10} {'Apr 1':>10}  {'Nov vs 90%':>10}")
print(f"  {'-'*52}")
for name, proj in projections.items():
    f_oct, f_nov, f_apr = get_fill(proj,oct1), get_fill(proj,nov1), get_fill(proj,apr1)
    vs90 = f_nov - 90.0
    print(f"  {name:<8} {f_oct:>9.1f}% {f_nov:>9.1f}% {f_apr:>9.1f}%  {'✅' if vs90>=0 else '⚠️ '} {vs90:>+.1f}pp")
print(f"\n  Inj rates — Low:{rates['Low']['inj']:,.0f}  Avg:{rates['Avg']['inj']:,.0f}  High:{rates['High']['inj']:,.0f} GWh/day")

In [ ]:
# Chart: storage projection + W-S spread
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
    subplot_titles=[f"Storage Fill % — {ANALYSIS_DATE} → {apr1}",
                    "Winter–Summer Spread (€/MWh)"],
    row_heights=[0.65, 0.35])

colors_proj = {'Low':'#d62728','Avg':'#2E75B6','High':'#2ca02c'}

# Historical fill
hist_fill = df_eu['full'].dropna()
fig.add_trace(go.Scatter(x=hist_fill.index, y=hist_fill,
    mode='lines', name='Actual', line=dict(color='black', width=1.5)), row=1, col=1)

# Projections
for name, proj in projections.items():
    fig.add_trace(go.Scatter(x=proj.index, y=proj/capacity*100,
        mode='lines', name=f'{name} injection',
        line=dict(color=colors_proj[name], width=2, dash='dash')), row=1, col=1)

# Key date lines
for d, label in [(oct1, f"Oct 1'{str(oct1.year)[-2:]}"),
                  (nov1, f"Nov 1'{str(nov1.year)[-2:]}"),
                  (apr1, f"Apr 1'{str(apr1.year)[-2:]}")]:
    fig.add_vline(x=pd.Timestamp(d), line_dash='dot', line_color='purple',
                  line_width=1, row=1, col=1)
    fig.add_annotation(x=pd.Timestamp(d), y=104, text=label,
                       showarrow=False, font=dict(size=8, color='purple'), row=1, col=1)

fig.add_hline(y=90, line_dash='dot', line_color='purple', line_width=1, row=1, col=1)

# W-S spread — customdata shows actual delivery windows per trade date
sw_filt = sw_spread[sw_spread.index >= START_DATE]
_sw_filt_hover = np.array([_ws_hover_label(dt) for dt in sw_filt.index])
fig.add_trace(go.Scatter(
    x=sw_filt.index, y=sw_filt,
    fill='tozeroy', fillcolor='rgba(46,117,182,0.12)',
    line=dict(color='#2E75B6', width=1.5), name='W–S Spread',
    customdata=_sw_filt_hover,
    hovertemplate='%{x|%d %b %Y}<br>%{customdata}<br>€%{y:+.2f}/MWh<extra></extra>',
), row=2, col=1)
fig.add_hline(y=0, line_color='black', line_width=0.8, row=2, col=1)
fig.add_hline(y=5, line_dash='dot', line_color='green', row=2, col=1)

fig.update_yaxes(ticksuffix='%', row=1, col=1)
fig.update_yaxes(title_text='€/MWh', row=2, col=1)
fig.update_layout(template='plotly_white', height=600,
                  hovermode='x unified',
                  legend=dict(orientation='h', yanchor='bottom', y=1.02, x=1, xanchor='right'))
fig.show()

## 5 · Winter Adequacy

In [ ]:
from src.analysis import SCENARIOS, run_depletion_model

# Projected fills at Nov 1
winter_starts = {name: get_fill(proj, nov1) * capacity / 100
                 for name, proj in projections.items()}

print(f"WINTER ADEQUACY — starting from projected Nov 1 fills")
print(f"{'='*72}")
print(f"  {'Inj scenario':<14} {'Nov 1 fill':>10}  {'Demand scenario':<18} {'End fill':>9} {'Depleted':>10}")
print(f"  {'-'*72}")

colors_sc = {'mild':'green','normal':'#2E75B6','cold':'orange','extreme':'red'}
depletion_results = {}

for inj_name, start_gwh in winter_starts.items():
    sf = start_gwh / capacity * 100
    for dem_key, scenario in SCENARIOS.items():
        dep      = run_depletion_model(start_gwh, scenario)
        end_gwh  = float(dep['storage_gwh'].iloc[-1])
        end_fill = end_gwh / capacity * 100
        depleted = dep['depleted'].any()
        depletion_results[(inj_name, dem_key)] = dep
        flag = '⚠️  Yes' if depleted else '✅ No'
        print(f"  {inj_name:<14} {sf:>9.1f}%  {scenario.label:<18} {end_fill:>8.1f}% {flag:>10}")

print(f"{'='*72}")
q4_p = float(curve_agg['Q4'].dropna().iloc[-1]) if 'Q4' in curve_agg.columns else None
q3_p = float(curve_agg['Q3'].dropna().iloc[-1]) if 'Q3' in curve_agg.columns else None
if q4_p and q3_p:
    print(f"\n  TTF Q4: €{q4_p:.2f}  Q3: €{q3_p:.2f}  Q4–Q3: €{q4_p-q3_p:+.2f}/MWh")
    print(f"  Curve: {'BACKWARDATION — winter risk priced' if q4_p>q3_p else 'CONTANGO — ample supply'}")

In [ ]:
# Depletion chart — 3 panels (one per injection scenario)
fig = make_subplots(rows=1, cols=3,
    subplot_titles=[f"{k} inj\n{winter_starts[k]/capacity*100:.1f}% at {nov1}"
                    for k in projections.keys()],
    shared_yaxes=True)

for col_idx, inj_name in enumerate(projections.keys(), 1):
    start_gwh = winter_starts[inj_name]
    for dem_key, scenario in SCENARIOS.items():
        dep  = depletion_results[(inj_name, dem_key)]
        fp   = dep['storage_gwh'] / capacity * 100
        real_dates = [pd.Timestamp(nov1) + pd.Timedelta(days=int(d)-1) for d in dep.index]
        fig.add_trace(go.Scatter(
            x=real_dates, y=fp, mode='lines',
            name=scenario.label,
            line=dict(color=colors_sc.get(dem_key,'grey'), width=2),
            legendgroup=scenario.label,
            showlegend=(col_idx==1),
        ), row=1, col=col_idx)
    fig.add_vline(x=pd.Timestamp(apr1), line_dash='dot',
                  line_color='purple', line_width=1, row=1, col=col_idx)
    fig.add_hline(y=20, line_dash='dot', line_color='darkred', row=1, col=col_idx)
    fig.add_hline(y=0,  line_color='black', line_width=0.4,    row=1, col=col_idx)

fig.update_yaxes(title_text='Fill (%)', ticksuffix='%', range=[-5,105], row=1, col=1)
fig.update_xaxes(tickformat='%d %b', tickangle=45)
fig.update_layout(
    title=f"Winter Depletion {nov1} → {apr1}  ·  3 injection × 4 demand scenarios",
    template='plotly_white', height=420, hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=-0.3, x=0.5, xanchor='center'))
fig.show()

## 6 · Interactive Forward Curve Tool

In [ ]:
# Plot forward curve at any date
# Change the date below to explore historical curve shapes

CURVE_DATE = None  # None = latest | or e.g. '2022-09-01'
plot_forward_curve(ttf, plot_date=CURVE_DATE)

## 7 · Export to PDF

In [ ]:
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib.colors import HexColor, white
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, PageBreak,
    Table, TableStyle, HRFlowable, Image
)
from reportlab.lib.enums import TA_CENTER
import tempfile, os

# ── Palette ───────────────────────────────────────────────────────────────────
NAVY     = HexColor("#0f172a"); BLUE    = HexColor("#2E75B6")
BLUE_LT  = HexColor("#dbeafe"); BLUE_ROW= HexColor("#EBF3FB")
GREY_BDR = HexColor("#e2e8f0"); SLATE   = HexColor("#64748b")
CHARCOAL = HexColor("#1e293b")

W, H   = letter
MARGIN = 0.75*inch
CW     = W - 2*MARGIN

ss = getSampleStyleSheet()
def sty(name, **kw): return ParagraphStyle(name, parent=ss["Normal"], **kw)
S = {
    "h1":   sty("h1",  fontName="Helvetica-Bold", fontSize=18, textColor=BLUE,
                       spaceBefore=0, spaceAfter=8, leading=24),
    "h2":   sty("h2",  fontName="Helvetica-Bold", fontSize=13, textColor=BLUE,
                       spaceBefore=14, spaceAfter=4, leading=17),
    "body": sty("body",fontName="Helvetica", fontSize=9.5, textColor=CHARCOAL,
                       spaceBefore=2, spaceAfter=4, leading=13),
    "small":sty("small",fontName="Helvetica", fontSize=8.5, textColor=SLATE,
                       spaceBefore=2, spaceAfter=3, leading=12),
    "cell": sty("cell",fontName="Helvetica", fontSize=8.5, textColor=CHARCOAL, leading=11),
    "hdr":  sty("hdr", fontName="Helvetica-Bold", fontSize=8.5,
                       textColor=HexColor("#1e3a5f"), leading=11),
    "ct":   sty("ct",  fontName="Helvetica-Bold", fontSize=28, textColor=white,
                       alignment=TA_CENTER, leading=36),
    "cs":   sty("cs",  fontName="Helvetica", fontSize=12,
                       textColor=HexColor("#94a3b8"), alignment=TA_CENTER, leading=16),
    "cm":   sty("cm",  fontName="Helvetica", fontSize=9,
                       textColor=HexColor("#64748b"), alignment=TA_CENTER, leading=12),
}

def sp(n=8): return Spacer(1,n)
def hr(): return HRFlowable(width="100%",thickness=0.5,color=GREY_BDR,spaceAfter=6,spaceBefore=6)
def h1(t): return [Paragraph(t,S["h1"]),
                   HRFlowable(width="100%",thickness=2,color=BLUE,spaceAfter=8,spaceBefore=2)]
def h2(t): return [Paragraph(t,S["h2"])]
def p(t):  return Paragraph(t,S["body"])
def sm(t): return Paragraph(t,S["small"])

def dtable(headers, rows, widths):
    data = [[Paragraph(h,S["hdr"]) for h in headers]]
    for ri,row in enumerate(rows):
        data.append([Paragraph(str(c),S["cell"]) for c in row])
    tbl = Table(data,colWidths=widths)
    cmds = [("BACKGROUND",(0,0),(-1,0),BLUE_LT),
            ("GRID",(0,0),(-1,-1),0.4,GREY_BDR),
            ("TOPPADDING",(0,0),(-1,-1),4),("BOTTOMPADDING",(0,0),(-1,-1),4),
            ("LEFTPADDING",(0,0),(-1,-1),6),("RIGHTPADDING",(0,0),(-1,-1),6),
            ("VALIGN",(0,0),(-1,-1),"TOP")]
    for ri in range(1,len(rows)+1):
        if ri%2==0: cmds.append(("BACKGROUND",(0,ri),(-1,ri),BLUE_ROW))
    tbl.setStyle(TableStyle(cmds))
    return [tbl,sp(8)]

FIGS = {}

def save_fig(fig, name, width=700, height=380):
    for trace in fig.data:
        if hasattr(trace, 'x') and trace.x is not None:
            trace.x = [str(x) for x in trace.x]
    tmp = tempfile.NamedTemporaryFile(suffix='.png', delete=False)
    fig.write_image(tmp.name, width=width, height=height, scale=2)
    FIGS[name] = tmp.name

def chart(name, width=CW, height=3.0*inch, caption=None):
    items = [Image(FIGS[name], width=width, height=height)]
    if caption: items.append(sm(f"<i>{caption}</i>"))
    items.append(sp(10))
    return items

def on_page(canvas, doc):
    canvas.saveState()
    canvas.setFillColor(NAVY)
    canvas.rect(MARGIN, H-0.50*inch, CW, 0.28*inch, fill=1, stroke=0)
    canvas.setFillColor(white); canvas.setFont("Helvetica-Bold", 7.5)
    canvas.drawString(MARGIN+6, H-0.37*inch, "EU Gas Storage & TTF Analysis")
    canvas.setFont("Helvetica", 7.5)
    canvas.drawRightString(W-MARGIN-4, H-0.37*inch,
                           f"Generated {ANALYSIS_DATE}")
    canvas.setStrokeColor(GREY_BDR); canvas.setLineWidth(0.4)
    canvas.line(MARGIN, 0.50*inch, W-MARGIN, 0.50*inch)
    canvas.setFillColor(SLATE); canvas.setFont("Helvetica", 7.5)
    canvas.drawRightString(W-MARGIN, 0.35*inch, f"Page {doc.page}")
    canvas.restoreState()

def on_first(canvas, doc):
    canvas.saveState()
    canvas.setFillColor(NAVY)
    canvas.rect(0, H-2.6*inch, W, 2.6*inch, fill=1, stroke=0)
    canvas.setFillColor(BLUE)
    canvas.rect(0, H-2.6*inch, W, 0.05*inch, fill=1, stroke=0)
    canvas.restoreState()

# ── Save charts ───────────────────────────────────────────────────────────────
import plotly.graph_objects as go

# Forward curve
latest_row = ttf.iloc[-1]
x_labels, y_vals = [], []
for m in range(1,13):
    col = f'M{m}'
    if col in latest_row.index and not pd.isna(latest_row[col]):
        dm  = (ttf.index[-1].month + m - 1) % 12 + 1
        yr  = ttf.index[-1].year + (ttf.index[-1].month + m - 1) // 12
        x_labels.append(f"{calendar.month_abbr[dm]}'{str(yr)[-2:]}")
        y_vals.append(float(latest_row[col]))
fig_curve = go.Figure()
fig_curve.add_trace(go.Scatter(x=x_labels, y=y_vals, mode='lines+markers',
    line=dict(color='#2E75B6',width=2.5), marker=dict(size=8)))
fig_curve.update_layout(title="TTF Forward Curve", template="plotly_white",
    xaxis_title="Delivery month", yaxis_title="€/MWh")
save_fig(fig_curve, 'curve')

# W-S spread
fig_sw = go.Figure()
fig_sw.add_trace(go.Scatter(x=sw_spread.index, y=sw_spread, mode='lines',
    fill='tozeroy', fillcolor='rgba(46,117,182,0.1)',
    line=dict(color='#2E75B6', width=1.5)))
fig_sw.add_hline(y=0, line_color='black', line_width=1)
fig_sw.update_layout(title="Winter–Summer Spread", template="plotly_white")
save_fig(fig_sw, 'sw')

# Rolling correlation
fig_corr = go.Figure()
fig_corr.add_trace(go.Scatter(x=corr.index, y=corr['corr_60d'],
    name='60d', line=dict(color='#2E75B6', width=1.5)))
fig_corr.add_hline(y=0, line_color='black', line_width=0.8)
fig_corr.update_layout(title="Rolling Correlation Fill % vs TTF M1",
    yaxis=dict(range=[-1.1,1.1]), template="plotly_white")
save_fig(fig_corr, 'corr')

# Projection
fig_proj = go.Figure()
hist_fill = df_eu['full'].dropna()
fig_proj.add_trace(go.Scatter(x=hist_fill.index, y=hist_fill,
    name='Actual', line=dict(color='black', width=1.5)))
colors_proj = {'Low':'#d62728','Avg':'#2E75B6','High':'#2ca02c'}
for name, proj in projections.items():
    fig_proj.add_trace(go.Scatter(x=proj.index, y=proj/capacity*100,
        name=name, line=dict(color=colors_proj[name], width=2, dash='dash')))
fig_proj.add_hline(y=90, line_dash='dot', line_color='purple')
fig_proj.update_layout(title="Storage Fill % — Projection", template="plotly_white",
    yaxis=dict(ticksuffix='%'))
save_fig(fig_proj, 'proj', height=400)

# Depletion
fig_dep = go.Figure()
for dem_key, scenario in SCENARIOS.items():
    dep  = depletion_results[('Avg', dem_key)]
    fp   = dep['storage_gwh'] / capacity * 100
    real = [pd.Timestamp(nov1) + pd.Timedelta(days=int(d)-1) for d in dep.index]
    fig_dep.add_trace(go.Scatter(x=real, y=fp, mode='lines',
        name=scenario.label,
        line=dict(color=colors_sc.get(dem_key,'grey'), width=2)))
fig_dep.add_hline(y=20, line_dash='dot', line_color='darkred')
fig_dep.update_layout(title="Winter Depletion (Avg injection scenario)",
    template="plotly_white", yaxis=dict(ticksuffix='%'))
save_fig(fig_dep, 'depletion')

print(f"✅ {len(FIGS)} charts saved — sections 1–5")

# ═══════════════════════════════════════════════════════════════════════
# New sections 6-8: Spreads · Volatility · LNG
# ═══════════════════════════════════════════════════════════════════════

from src.analysis.spread_analysis import build_spread_matrix
from src.analysis.price_analysis import (
    rolling_volatility, seasonal_price_pattern, price_distribution_by_fill
)

# ── Section 6: Time Spread Analysis ──────────────────────────────────────────

# W-S spread history (dedicated PDF chart)
fig_sw_hist = go.Figure()
fig_sw_hist.add_trace(go.Scatter(
    x=sw_spread.index, y=sw_spread, mode='lines', fill='tozeroy',
    fillcolor='rgba(46,117,182,0.12)', line=dict(color='#2E75B6', width=1.5)
))
fig_sw_hist.add_hline(y=0, line_color='black', line_width=1)
fig_sw_hist.add_hline(y=5, line_dash='dot', line_color='green',
    annotation_text='~€5/MWh breakeven')
fig_sw_hist.update_layout(
    title='Winter–Summer Spread History (€/MWh)',
    template='plotly_white', yaxis_title='€/MWh'
)
save_fig(fig_sw_hist, 'sw_hist')

# M1-M12 spread vs fill scatter with polynomial fit
if 'M1' in ttf.columns and 'M12' in ttf.columns:
    _m1m12   = (ttf['M1'] - ttf['M12']).rename('spread')
    _fill_s  = df_eu_full['full'].tz_localize(None).reindex(_m1m12.index).ffill()
    _scat    = pd.DataFrame({'spread': _m1m12, 'fill': _fill_s}).dropna()
    _coef    = np.polyfit(_scat['fill'], _scat['spread'], 2)
    _xfit    = np.linspace(_scat['fill'].min(), _scat['fill'].max(), 120)
    _yfit    = np.polyval(_coef, _xfit)
    _roots   = np.roots(_coef)
    _rreal   = _roots[np.isreal(_roots)].real
    _in_rng  = _rreal[(_rreal >= _scat['fill'].min()) & (_rreal <= _scat['fill'].max())]
    fig_scat = go.Figure()
    fig_scat.add_trace(go.Scatter(
        x=_scat['fill'], y=_scat['spread'], mode='markers',
        marker=dict(color='#2E75B6', size=3, opacity=0.35), name='Daily'
    ))
    fig_scat.add_trace(go.Scatter(
        x=_xfit, y=_yfit, mode='lines',
        line=dict(color='#d62728', width=2.5), name='Poly fit'
    ))
    fig_scat.add_hline(y=0, line_color='black', line_width=1)
    if len(_in_rng) > 0:
        _zc = float(sorted(_in_rng, key=lambda v: abs(v - _scat['fill'].mean()))[0])
        fig_scat.add_vline(x=_zc, line_color='purple', line_dash='dot',
            annotation_text=f'Zero-crossing: {_zc:.1f}%', annotation_position='top left')
    else:
        _zc = None
    fig_scat.update_layout(
        title='M1–M12 Spread vs EU Fill Rate',
        template='plotly_white',
        xaxis_title='Fill Rate (%)', yaxis_title='€/MWh (M1 – M12)'
    )
    save_fig(fig_scat, 'spread_scatter')
else:
    _scat = None; _zc = None

# ── Section 7: Volatility ─────────────────────────────────────────────────────
if 'M1' in ttf.columns:
    _vol = rolling_volatility(ttf['M1'], windows=[21, 63])
    _vol_fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
        subplot_titles=['TTF M1 Rolling Volatility (annualised %)', 'EU Fill Rate (%)'],
        row_heights=[0.65, 0.35]
    )
    _vol_fig.add_trace(go.Scatter(
        x=_vol.index, y=_vol['vol_21d'], name='21d vol',
        line=dict(color='#f97316', width=1.5)
    ), row=1, col=1)
    _vol_fig.add_trace(go.Scatter(
        x=_vol.index, y=_vol['vol_63d'], name='63d vol',
        line=dict(color='#2E75B6', width=1.5)
    ), row=1, col=1)
    _fill_for_vol = df_eu['full'].dropna()
    _vol_fig.add_trace(go.Scatter(
        x=_fill_for_vol.index, y=_fill_for_vol, fill='tozeroy',
        line=dict(color='#22c55e', width=1), fillcolor='rgba(34,197,94,0.15)',
        name='Fill %'
    ), row=2, col=1)
    _vol_fig.update_layout(template='plotly_white', title='TTF M1 Volatility & EU Storage Fill')
    save_fig(_vol_fig, 'vol')

    _patt    = seasonal_price_pattern(ttf, col='M1')
    _bymo    = _patt['by_month'].copy()
    _mo_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    _bymo.index = [_mo_names[i-1] for i in _bymo.index]
    _mo_colors = ['#d62728' if m in ['Dec','Jan','Feb','Nov'] else '#2E75B6'
                  for m in _bymo.index]
    fig_seas = go.Figure(go.Bar(
        x=list(_bymo.index), y=_bymo['avg_price'], marker_color=_mo_colors
    ))
    fig_seas.update_layout(
        template='plotly_white', title='Average TTF M1 Price by Calendar Month',
        yaxis_title='€/MWh', margin=dict(t=50)
    )
    save_fig(fig_seas, 'seasonal', height=300)
else:
    _vol = None

# ── Section 8: LNG Storage ────────────────────────────────────────────────────
LNG_PATH = Path("data/processed/eu_lng_full.parquet")
lng_df = None
if LNG_PATH.exists():
    lng_df = pd.read_parquet(LNG_PATH)
    lng_df.index = pd.to_datetime(lng_df.index).tz_localize(None)
    lng_df = lng_df.sort_index()

    _gas_for_lng = df_eu_full[['gasInStorage','full']].copy()
    _gas_for_lng.index = pd.to_datetime(_gas_for_lng.index).tz_localize(None)

    _merged_lng = (
        lng_df[['full']].rename(columns={'full': 'lng_full'})
        .join(_gas_for_lng[['full']].rename(columns={'full': 'gas_full'}), how='inner')
        .dropna()
    )

    from plotly.subplots import make_subplots as _ms2
    fig_lng_dual = _ms2(specs=[[{"secondary_y": True}]])
    fig_lng_dual.add_trace(go.Scatter(
        x=_merged_lng.index, y=_merged_lng['gas_full'],
        name='Gas Fill %', line=dict(color='#22c55e', width=2)
    ), secondary_y=False)
    fig_lng_dual.add_trace(go.Scatter(
        x=_merged_lng.index, y=_merged_lng['lng_full'],
        name='LNG Fill %', line=dict(color='#0ea5e9', width=2)
    ), secondary_y=True)
    fig_lng_dual.update_layout(
        template='plotly_white',
        title='EU Gas Storage Fill vs EU LNG Fill Rate (%)',
        legend=dict(orientation='h', y=-0.15)
    )
    fig_lng_dual.update_yaxes(title_text='Gas Fill (%)', secondary_y=False)
    fig_lng_dual.update_yaxes(title_text='LNG Fill (%)', secondary_y=True)
    save_fig(fig_lng_dual, 'lng_dual')

    _gas_twh = _gas_for_lng[['gasInStorage']].rename(columns={'gasInStorage': 'gas_TWh'})
    _lng_twh = lng_df[['inventory']].rename(columns={'inventory': 'lng_TWh'})
    _combined = _gas_twh.join(_lng_twh, how='inner').dropna()
    _combined['total_TWh'] = _combined['gas_TWh'] + _combined['lng_TWh']
    fig_buffer = go.Figure()
    fig_buffer.add_trace(go.Scatter(
        x=_combined.index, y=_combined['total_TWh'],
        name='Total (Gas + LNG)', fill='tozeroy',
        line=dict(color='#7c3aed', width=2),
        fillcolor='rgba(124,58,237,0.10)'
    ))
    fig_buffer.add_trace(go.Scatter(
        x=_combined.index, y=_combined['gas_TWh'],
        name='Gas Storage', line=dict(color='#22c55e', width=1.5)
    ))
    fig_buffer.add_trace(go.Scatter(
        x=_combined.index, y=_combined['lng_TWh'],
        name='LNG', line=dict(color='#0ea5e9', width=1.5)
    ))
    fig_buffer.update_layout(
        template='plotly_white',
        title='Combined EU Energy Buffer: Gas + LNG (TWh)',
        yaxis_title='TWh',
        legend=dict(orientation='h', y=-0.15)
    )
    save_fig(fig_buffer, 'lng_buffer')
    print(f"✅ LNG data loaded: {len(lng_df)} rows | {lng_df.index.min().date()} → {lng_df.index.max().date()}")
else:
    print("⚠️  eu_lng_full.parquet not found — Section 8 will be skipped in PDF")

print(f"✅ {len(FIGS)} charts saved total")


In [ ]:
from pathlib import Path

story = []

# ── Cover ──────────────────────────────────────────────────────────────────────
story += [sp(2.7*inch),
    Paragraph("EU Gas Storage & TTF", S["ct"]), sp(8),
    Paragraph("Market Analysis Report", S["cs"]), sp(6),
    Paragraph(f"As of {latest_date.strftime('%d %B %Y')}  ·  EU Fill Rate: {fill_pct:.1f}%", S["cm"]),
    sp(20),
    HRFlowable(width="30%",thickness=1,color=BLUE,spaceAfter=14,hAlign="CENTER"),
    Paragraph("Includes: Storage · TTF Curve · Spreads · Volatility · LNG", S["cm"]),
    sp(6), Paragraph(f"Generated {ANALYSIS_DATE}", S["cm"]),
    PageBreak()]

# ── 1. Storage snapshot ────────────────────────────────────────────────────────
story += h1("1  Storage Snapshot")
story += dtable(
    ["Metric","Value","Context"],
    [
        ["Fill rate",       f"{fill_pct:.1f}%",
         "EU target: 90% by Nov 1"],
        ["Storage",         f"{current_gwh/1000:,.1f} TWh",
         f"Capacity: {capacity/1000:,.1f} TWh"],
        ["Analysis date",   str(ANALYSIS_DATE),
         f"Latest data point"],
        ["Inj P10/P50/P90", f"{rates['Low']['inj']:,.0f} / {rates['Avg']['inj']:,.0f} / {rates['High']['inj']:,.0f} GWh/day",
         "Historical active injection days (Apr–Oct)"],
    ],
    [2.0*inch, 1.8*inch, 3.0*inch]
)

# ── 2. Forward curve ───────────────────────────────────────────────────────────
story.append(PageBreak())
story += h1("2  TTF Forward Curve")
story += [p(
    "The TTF forward curve shows the market price for gas delivered in each future month. "
    "Backwardation (M1 > M12) signals current tightness. "
    "The Winter–Summer spread indicates whether injection economics are viable."
), sp(6)]
story += chart('curve', caption="TTF forward curve — M1 to M12. Colours: red=winter, green=spring, orange=summer, purple=autumn.")
story += h2("Winter–Summer Spread")
story += chart('sw', height=2.5*inch, caption="Winter–Summer spread (Q4+Q1 minus Q2+Q3). Above €5/MWh = injection economic.")

sw_val = float(sw_spread.iloc[-1])
q4_p   = float(curve_agg['Q4'].dropna().iloc[-1]) if 'Q4' in curve_agg.columns else None
q3_p   = float(curve_agg['Q3'].dropna().iloc[-1]) if 'Q3' in curve_agg.columns else None
if q4_p and q3_p:
    story += [Table([[p(f"W–S spread: €{sw_val:.2f}/MWh  |  Q4: €{q4_p:.2f}  Q3: €{q3_p:.2f}  |  "
                        f"{'BACKWARDATION — winter risk priced' if q4_p>q3_p else 'CONTANGO — ample supply'}")
    ]],colWidths=[CW],style=[
        ("BACKGROUND",(0,0),(-1,-1),BLUE_LT),
        ("LINEBEFORE",(0,0),(0,-1),3,BLUE),
        ("TOPPADDING",(0,0),(-1,-1),7),("BOTTOMPADDING",(0,0),(-1,-1),7),
        ("LEFTPADDING",(0,0),(-1,-1),10),("RIGHTPADDING",(0,0),(-1,-1),8),
    ]), sp(8)]

# ── 3. Correlation ─────────────────────────────────────────────────────────────
story.append(PageBreak())
story += h1("3  Storage vs. TTF Correlation")
story += [p(
    "Rolling 60-day Pearson correlation between EU fill rate and TTF M1. "
    "Below -0.7 = storage is the dominant price driver."
), sp(6)]
story += chart('corr', caption="Rolling 60-day correlation. Red dotted = -0.7 threshold.")
story += dtable(
    ["Parameter","Value","Interpretation"],
    [
        ["Model",        "log(TTF M1) = α + β × fill%",        "Semi-log OLS"],
        ["β (slope)",    f"{reg['slope']:.4f}",                 f"+1pp fill = {reg['slope']*100:.2f}% price change"],
        ["R²",           f"{reg['r_squared']:.3f}",             f"Fill explains {reg['r_squared']*100:.1f}% of variance"],
        ["60d corr",     f"{float(corr['corr_60d'].dropna().iloc[-1]):.2f}",
         "Current regime"],
    ],
    [1.8*inch, 1.5*inch, 3.5*inch]
)

# ── 4. Injection pace ──────────────────────────────────────────────────────────
story.append(PageBreak())
story += h1("4  Injection Pace & Nov 1 Projection")
story += [p(
    "Projected fill at Oct 1 / Nov 1 / Apr 1 under three injection scenarios "
    "(P10 / P50 / P90 of historical active injection days, Apr–Oct)."
), sp(6)]
story += chart('proj', caption="Storage fill % — historical (black) and 3 injection scenarios (dashed). Purple = 90% target.")
story += dtable(
    ["Scenario","Oct 1","Nov 1","Apr 1","vs 90%"],
    [[name,
      f"{get_fill(proj,oct1):.1f}%",
      f"{get_fill(proj,nov1):.1f}%",
      f"{get_fill(proj,apr1):.1f}%",
      f"{'✅' if get_fill(proj,nov1)-90>=0 else '⚠️'} {get_fill(proj,nov1)-90:+.1f}pp"]
     for name, proj in projections.items()],
    [1.5*inch, 1.3*inch, 1.3*inch, 1.3*inch, 1.3*inch]
)

# ── 5. Winter adequacy ─────────────────────────────────────────────────────────
story.append(PageBreak())
story += h1("5  Winter Adequacy")
story += [p(
    "Depletion model from projected Nov 1 fills (Avg injection scenario) "
    "across four demand scenarios. Red dotted = 20% critical level."
), sp(6)]
story += chart('depletion', caption=f"Winter depletion {nov1} → {apr1}. Avg injection starting point.")


# ── 6. Time Spread Analysis ────────────────────────────────────────────────────
story.append(PageBreak())
story += h1("6  Time Spread Analysis")
story += [p(
    "The TTF calendar spread (M1–M12) is the primary signal of storage value. "
    "When the curve is in contango (spread &lt; 0), storage injection earns a forward premium. "
    "When in backwardation (spread &gt; 0), near-term supply is tight and injection economics weaken. "
    "The approximate injection breakeven is €5/MWh for the Winter–Summer spread."
), sp(6)]

if 'sw_hist' in FIGS:
    story += chart('sw_hist',
        caption="Winter–Summer spread (Winter Q4+Q1 minus Summer Q2+Q3). "
                "Green dotted = ~€5/MWh injection breakeven. Below zero = backwardation.")

if 'spread_scatter' in FIGS:
    story += h2("M1–M12 Spread vs EU Fill Rate")
    story += chart('spread_scatter', height=2.8*inch,
        caption="Each dot = one trading day. Red curve = quadratic polynomial fit. "
                "Purple line = zero-crossing inflection point (market switches contango/backwardation).")

# Median spread by fill quintile
if 'M1' in ttf.columns and 'M12' in ttf.columns:
    _m1m12q  = (ttf['M1'] - ttf['M12']).rename('spread')
    _fillq   = df_eu_full['full'].tz_localize(None).reindex(_m1m12q.index).ffill()
    _qtable  = pd.DataFrame({'spread': _m1m12q, 'fill': _fillq}).dropna()
    _qtable['quintile'] = pd.qcut(
        _qtable['fill'], q=5,
        labels=['Q1 (lowest fill)','Q2','Q3','Q4','Q5 (highest fill)']
    )
    _qstats = _qtable.groupby('quintile', observed=True)['spread'].agg(
        n='count',
        p25=lambda x: x.quantile(0.25),
        median='median',
        p75=lambda x: x.quantile(0.75)
    ).reset_index()
    story += h2("Median M1–M12 Spread by Fill Quintile")
    story += dtable(
        ["Fill Quintile","N days","P25 (€/MWh)","Median (€/MWh)","P75 (€/MWh)","Regime"],
        [[str(row['quintile']), str(int(row['n'])), f"{row['p25']:.2f}",
          f"{row['median']:.2f}", f"{row['p75']:.2f}",
          "Backwardation" if row['median'] > 0 else "Contango"]
         for _, row in _qstats.iterrows()],
        [1.65*inch, 0.65*inch, 1.0*inch, 1.1*inch, 1.0*inch, 1.35*inch]
    )
    _inflect_row = _qstats.iloc[(_qstats['median'].abs()).values.argmin()]
    _inflect_q   = str(_inflect_row['quintile'])
    story += [p(
        f"<b>Key insight:</b> The spread zero-crossing typically falls in the "
        f"{_inflect_q} fill range (median spread: €{_inflect_row['median']:.2f}/MWh). "
        "Below this level the curve tends toward backwardation (winter scarcity priced); "
        "above it, toward contango (ample supply, injection economics intact)."
    ), sp(6)]

# ── 7. TTF Price Volatility ────────────────────────────────────────────────────
story.append(PageBreak())
story += h1("7  TTF Price Volatility")
story += [p(
    "Annualised rolling volatility of the TTF M1 front-month price. "
    "21-day vol captures short-term stress episodes; "
    "63-day vol measures the broader volatility regime. "
    "Elevated vol during injection season often precedes spread compression as traders hedge."
), sp(6)]

if 'vol' in FIGS:
    story += chart('vol',
        caption="21d (orange) and 63d (blue) annualised rolling volatility (%) with EU fill rate overlay (green).")

if 'M1' in ttf.columns:
    from src.analysis.price_analysis import price_distribution_by_fill as _pdbf
    _dist = _pdbf(ttf, df_eu_full.assign(
        full=df_eu_full['full']
    ), n_buckets=5)
    story += h2("Price Distribution by Fill Rate Bucket")
    story += dtable(
        ["Fill Bucket","N days","Mean (€/MWh)","Median","P10","P90"],
        [[row['bucket_label'], str(int(row['count'])), f"{row['mean']:.1f}",
          f"{row['median']:.1f}", f"{row['p10']:.1f}", f"{row['p90']:.1f}"]
         for _, row in _dist.reset_index().iterrows()],
        [1.5*inch, 0.65*inch, 1.15*inch, 0.95*inch, 0.85*inch, 0.85*inch]
    )

if 'seasonal' in FIGS:
    story += h2("Seasonal Average Price by Month")
    story += chart('seasonal', height=2.5*inch,
        caption="Average TTF M1 settlement price by calendar month across all history. "
                "Red bars = winter months (Nov–Feb).")

# ── 8. LNG Storage ─────────────────────────────────────────────────────────────
if lng_df is not None:
    story.append(PageBreak())
    story += h1("8  EU LNG Storage")
    story += [p(
        "EU LNG terminal inventories from GIE ALSI+. "
        "LNG send-out provides a flexible supply buffer supplementing pipeline imports and underground storage. "
        "The combined gas + LNG energy buffer gives the broadest view of EU supply security heading into winter."
    ), sp(6)]

    if 'lng_dual' in FIGS:
        story += h2("LNG Fill Rate vs Gas Storage Fill Rate")
        story += chart('lng_dual',
            caption="EU gas storage fill % (green, left axis) vs EU LNG terminal fill % (blue, right axis). "
                    "Divergence often reflects LNG vessel scheduling or seasonal send-out patterns.")

    if 'lng_buffer' in FIGS:
        story += h2("Combined Energy Buffer")
        story += chart('lng_buffer',
            caption="Total EU energy buffer: gas storage (green) + LNG terminals (blue) = combined (purple). "
                    "Combined buffer provides a more complete picture of winter supply security than gas storage alone.")

    _lng_clean = lng_df.dropna(subset=['full'])
    if _lng_clean.empty:
        print('⚠️  LNG fill data empty — skipping LNG stats table')
    else:
        _lat_lng = _lng_clean.iloc[-1]
        _send    = _lat_lng.get('sendOut', float('nan'))
        _dtmi    = _lat_lng.get('dtmi', float('nan'))
        _lat_gas = df_eu_full['gasInStorage'].tz_localize(None).dropna().iloc[-1]
        story += h2("Latest LNG Statistics")
        story += dtable(
            ["Metric","Value","Note"],
            [
                ["LNG Fill Rate",       f"{_lat_lng['full']:.1f}%",
                 f"As of {_lng_clean.index[-1].date()}"],
                ["LNG Inventory",       f"{_lat_lng.get('inventory', float('nan')):.1f} TWh",
                 "Working gas in EU LNG terminals"],
                ["Gas Storage",         f"{_lat_gas:.1f} TWh",
                 "EU underground gas storage"],
                ["Combined Buffer",     f"{_lat_lng.get('inventory', 0.0) + _lat_gas:.1f} TWh",
                 "LNG + gas storage total"],
                ["Send-Out Rate",
                 f"{_send:,.0f} GWh/day" if pd.notna(_send) else "N/A",
                 "Regasified gas sent to grid"],
                ["LNG Capacity (dtmi)",
                 f"{_dtmi:.1f} TWh" if pd.notna(_dtmi) else "N/A",
                 "Declared total maximum inventory"],
            ],
            [2.0*inch, 1.6*inch, 3.1*inch]
        )
else:
    story += [sp(10), p(
        "<i>⚠️ Section 8 (LNG) skipped — "
        "<b>data/processed/eu_lng_full.parquet</b> not found. "
        "Run notebook 11 to generate LNG data, then re-run this cell.</i>"
    ), sp(6)]

# ── Footer ─────────────────────────────────────────────────────────────────────
story += [sp(20), hr(),
    sm(f"Data: GIE AGSI+ (storage) · Databento NDEX.IMPACT (TTF) · GIE ALSI+ (LNG) · "
       f"eu-gas-storage-analysis · Generated {ANALYSIS_DATE}")]

# ── Build PDF ──────────────────────────────────────────────────────────────────
Path("data/processed").mkdir(parents=True, exist_ok=True)
OUT = "data/processed/gas_storage_ttf_report.pdf"
doc = SimpleDocTemplate(OUT, pagesize=letter,
    leftMargin=MARGIN, rightMargin=MARGIN,
    topMargin=0.78*inch, bottomMargin=0.68*inch,
    title="EU Gas Storage & TTF Analysis")
doc.build(story, onFirstPage=on_first, onLaterPages=on_page)

# Cleanup
for f in FIGS.values():
    try: os.unlink(f)
    except: pass

print(f"✅ Report: {OUT}")
print(f"   Size  : {Path(OUT).stat().st_size/1024:.0f} KB")


In [ ]:
from IPython.display import FileLink, display
display(FileLink("data/processed/gas_storage_ttf_report.pdf",
                 result_html_prefix="📄 Download: "))
